<a href="https://colab.research.google.com/github/AngelTroncoso/Transformers/blob/main/HugginFace_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1 - Pipeline

In [ ]:
!pip install transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 42.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 21.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 69.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.9 MB/s eta 0:00:00


[Librería Transformers](https://github.com/huggingface/transformers)

In [ ]:
from transformers import pipeline

In [ ]:
# tarea de calsficiación
classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


Xformers is not installed correctly. If you want to use memory_efficient_attention to accelerate training use the following command to install Xformers
pip install xformers.


In [ ]:
res = classifier("Me encantan las clases de Nechu, explica genial")
print(res)

[{'label': 'POSITIVE', 'score': 0.8972373008728027}]


In [ ]:
res = classifier("El profesor es bueno todo será sencillo")
print(res)

[{'label': 'NEGATIVE', 'score': 0.6551783084869385}]


#### Selección del modelo

In [ ]:
# seleccionamos el mismo modelo que tenemos por defecto
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

In [ ]:
res = classifier("El profesor es bueno todo será sencillo")
print(res)

[{'label': 'NEGATIVE', 'score': 0.6551783084869385}]


Existe una amplia variedad de 'pipelines': [lista de pipelines](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.pipeline)

### 2 - Modelo y Tokenizer

El modelo lo hemos ejecutado con apenas una línea, pero realmente hay bastantes etapas que ocurren por debajo. En el siguiente código vamos a ver las más importantes.

In [ ]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# Primero veamos cuales son las etapas anteriores con el mismo modelo
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
model_name = "pysentimiento/robertuito-sentiment-analysis"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

res = classifier("El profesor es bueno todo será sencillo")
print(res)

[{'label': 'POS', 'score': 0.9181905388832092}]


### 3 - ¿Para qué sirve el tokenizer?

Codificación (antes del LLM)

In [ ]:
secuencia = "El profesor es bueno todo será sencillo"
res = tokenizer(secuencia)
print(res)

{'input_ids': [0, 459, 5934, 442, 1220, 658, 1504, 9764, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


Paso a paso

In [ ]:
tokens = tokenizer.tokenize(secuencia)
print(tokens)

['▁el', '▁profesor', '▁es', '▁bueno', '▁todo', '▁será', '▁sencillo']


In [ ]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids)

[459, 5934, 442, 1220, 658, 1504, 9764]


Decodificar (después del LLM

In [ ]:
tokenizer.decode(res['input_ids'])

'<s> el profesor es bueno todo será sencillo</s>'

### 4 - Guardar modelo y tokenizer en local

In [ ]:
model_path = ("./modelo")
tokenizer.save_pretrained(model_path)
model.save_pretrained(model_path)

In [ ]:
tokenizer_local = AutoTokenizer.from_pretrained(model_path)
model_local = AutoModelForSequenceClassification.from_pretrained(model_path)

### 5 - Pytorch

También compatible con tensorflow

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
sentences = [
    "He estado deseando un curso así toda mi vida",
    "Me encanta HuggingFace"
]

Tokenizer

In [ ]:
batch = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)
print(batch)

Modelo

In [ ]:
with torch.no_grad():
    outputs = model(**batch)
    predictions = F.softmax(outputs.logits, dim=1)
    labels = torch.argmax(predictions, dim=1)

In [ ]:
# Logits
print(outputs)

In [ ]:
# Neg, Neu, Pos
print(predictions)

In [ ]:
# etiquetas
print(labels)

In [ ]:
model